# K-Means Experiments (Thesis-aligned)

Notebook terbagi menjadi bagian-bagian sesuai rencana tesis: impor library, penentuan K (Elbow + Silhouette), pelatihan model K-Means final, analisis cluster, dan jalankan eksperimen per-variant (baseline vs PCA).

In [ ]:
# Part 1 — Import libraries and configuration
import numpy as np
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score
import pandas as pd
import gc

# Configuration (edit paths if needed)
INPUT_DIR = Path("/media/bioinfo04/Expansion/2427051003_dataset_vector")
# If you used after_bert_pipeline variants, point to the chosen folder e.g. _pca256 or normalized
VARIANT = 'pca256'  # options: 'baseline', 'normalized', 'pca256', 'pca128'
COMBINED_FILE = Path('combined_embeddings.npy')  # optional combined file if available
RANDOM_STATE = 42
SAMPLE_FOR_METRICS = 200000  # sample size for silhouette if dataset is huge (set lower if needed)

In [ ]:
# Helper: load embeddings depending on variant
def find_files_for_variant(variant='baseline'):
    if COMBINED_FILE.exists():
        return [COMBINED_FILE]
    patterns = {
        'baseline': '*_embeddings.npy',
        'normalized': '*_normalized_embeddings.npy',
        'pca256': '*_pca256_embeddings.npy',
        'pca128': '*_pca128_embeddings.npy',
    }
    pat = patterns.get(variant, '*_embeddings.npy')
    return sorted(list(INPUT_DIR.glob(pat)))

def load_concat(files, mmap=False):
    # Load and vertically stack embeddings; use mmap where possible to avoid large memory use
    arrays = []
    for f in files:
        if mmap:
            arr = np.load(f, mmap_mode='r')
            arrays.append(np.array(arr))  # convert chunk-by-chunk to avoid memmap pitfalls later
        else:
            arrays.append(np.load(f))
    if len(arrays) == 0:
        raise FileNotFoundError('No embedding files found')
    return np.vstack(arrays)

# Quick check files
files = find_files_for_variant(VARIANT)
print(f'Found {len(files)} files for variant 
')
for f in files[:10]:
    print(' -', f.name)

## Part 2 — Determine K (Elbow + Silhouette)
Rencana: gunakan Elbow (inertia) untuk range K, dan silhouette score (sample jika dataset besar). Jika dataset sangat besar gunakan `MiniBatchKMeans` untuk uji cepat.

In [ ]:
# Part 2 — compute inertia and silhouette for K range
from sklearn.utils import resample

K_RANGE = range(2, 16)
inertia = []
sil_scores = []

# Load a sample if dataset too large for memory — user can change mmap flag
print('Loading embeddings (may be large)...')
emb_files = find_files_for_variant(VARIANT)
embeddings = load_concat(emb_files, mmap=False)
n_samples = embeddings.shape[0]
print(f'Loaded embeddings shape: {embeddings.shape}')

# If huge, sample for silhouette calculation
if n_samples > SAMPLE_FOR_METRICS:
    sample_idx = np.random.RandomState(RANDOM_STATE).choice(n_samples, SAMPLE_FOR_METRICS, replace=False)
    sample_for_sil = embeddings[sample_idx]
else:
    sample_for_sil = embeddings

for k in K_RANGE:
    print(f'-- Testing K={k}')
    # Use MiniBatchKMeans for speed; use full KMeans for final training
    km = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE, batch_size=4096)
    km.fit(embeddings)
    inertia.append(km.inertia_)
    # silhouette on sample only
    labels_sample = km.predict(sample_for_sil)
    sil = silhouette_score(sample_for_sil, labels_sample) if len(np.unique(labels_sample))>1 else -1
    sil_scores.append(sil)

# Plot results
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,2, figsize=(14,4))
ax[0].plot(list(K_RANGE), inertia, '-o')
ax[0].set_xlabel('K')
ax[0].set_ylabel('Inertia')
ax[0].set_title('Elbow: Inertia vs K')
ax[1].plot(list(K_RANGE), sil_scores, '-o')
ax[1].set_xlabel('K')
ax[1].set_ylabel('Silhouette Score (sample)')
ax[1].set_title('Silhouette (sample) vs K')
plt.show()

## Part 3 — Train final K-Means and save model
Pilih K berdasarkan hasil Part 2. Untuk dataset besar gunakan `MiniBatchKMeans` atau `KMeans` dengan `n_init` lebih kecil untuk stabilitas/performance.

In [ ]:
# Part 3 — Fit final model and save
CHOSEN_K = 4  # <-- replace with your chosen K from previous cell
USE_MINIBATCH = True  # set False to use sklearn.KMeans (may be slower)

if USE_MINIBATCH:
    model = MiniBatchKMeans(n_clusters=CHOSEN_K, random_state=RANDOM_STATE, batch_size=4096, n_init=10)
else:
    model = KMeans(n_clusters=CHOSEN_K, random_state=RANDOM_STATE, n_init=10)

print('Fitting final KMeans on full dataset...')
model.fit(embeddings)
labels = model.labels_
centroids = model.cluster_centers_

# Save model and labels
out_model = Path('model_kmeans_log.pkl')
joblib.dump(model, out_model)
np.save('cluster_labels.npy', labels)
np.save('cluster_centroids.npy', centroids)
print(f'Saved model: {out_model}, labels and centroids.')

## Part 4 — Cluster analysis & sampling
Tunjukkan ukuran cluster, beberapa contoh log per cluster (jika mapping ke log asli tersedia), dan simpan ringkasan ke CSV untuk analisa lebih lanjut.

In [ ]:
# Part 4 — Analyze clusters
from collections import Counter
counts = Counter(labels)
print('Cluster sizes:')
for k, c in sorted(counts.items()):
    print(f' - Cluster {k}: {c} samples')

# If you have original logs aligned to embeddings, load mapping here and sample examples.
# Example: assume a CSV with original logs in same order as embeddings: logs.csv with column 'log'
logs_csv = Path('../dataset/logs_in_order.csv')  # optional: provide a file mapping
if logs_csv.exists():
    df_logs = pd.read_csv(logs_csv)
    df_logs['cluster'] = labels
    # save cluster summary
    summary_path = Path('cluster_summary.csv')
    df_logs.to_csv(summary_path, index=False)
    print('Saved cluster summary to', summary_path)
else:
    print('No original logs mapping found at', logs_csv, '. Showing indices for samples instead.')
    for k in range(CHOSEN_K):
        idxs = np.where(labels==k)[0][:5]
        print(f' Cluster {k} sample indices: {list(idxs)}')

## Part 5 — Compare variants (optional)
Jika Anda ingin membandingkan Baseline vs PCA256 performance, jalankan eksperimen kecil: hitung silhouette pada subset sama untuk kedua variant dan bandingkan.

In [ ]:
# Part 5 — Quick comparison function (Baseline vs PCA variant)
def compare_variants(baseline_files, variant_files, sample_size=50000):
    # load sample from baseline and variant (matching random indices)
    base = load_concat(baseline_files)
    var = load_concat(variant_files)
    n = min(base.shape[0], var.shape[0], sample_size)
    idx = np.random.RandomState(RANDOM_STATE).choice(base.shape[0], n, replace=False)
    base_s = base[idx]
    var_s = var[idx]
    # cluster with same K (small K for speed)
    k = 10
    km_base = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE).fit(base_s)
    km_var = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE).fit(var_s)
    sil_base = silhouette_score(base_s, km_base.labels_)
    sil_var = silhouette_score(var_s, km_var.labels_)
    return sil_base, sil_var

# Example usage (uncomment and set correct patterns):
# base_files = sorted(list(Path('/media/.../dataset_vector').glob('*_embeddings.npy')))
# pca_files = sorted(list(Path('/media/.../dataset_vector_pca256').glob('*_pca256_embeddings.npy')))
# print(compare_variants(base_files, pca_files, sample_size=20000))

## Part 6 — Thesis notes and recommended settings
- Untuk dataset besar (thesis): gunakan `VARIANT='pca256'` dan `MiniBatchKMeans` untuk eksperimen awal.
- Simpan model `pca_model_256.pkl` dari `after_bert_pipeline.py` dan gunakan itu untuk inference pada log baru.
- Untuk evaluasi, gunakan `silhouette_score`, `Davies-Bouldin`, dan inspeksi manual sample dari tiap cluster.